# 00 — Feasibility check (Kaggle, T4 16 GB)

Run on Kaggle with **Accelerator: GPU T4** and **Internet: on**. Budget: under 30 GPU minutes.

Each step runs as a separate Python process. It writes its result to `results/<step>.json`. If a step crashes, it is recorded as FAIL with the error, and the next steps still run. The last cell builds `results/feasibility.md` from those JSON files only.

## 0. Config — every seed and name used below comes from here

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

CONFIG = {
    # "all", or a list of step names to rerun only those, e.g. ["step3_libero", "step5_lora"].
    # The install cell always runs. Skipped steps show as SKIPPED in feasibility.md.
    "steps_to_run": ["step3_libero", "step5_lora"],
    "seed": 1000,
    "lerobot_commit": "b07adcc5e3ce42c26a248236dbb3571f84c28c43",  # LeRobot main on 2026-11-25; same pin for local setup
    "uv_version": "0.13.7",
    "peft_version": "0.21.2",
    "libero_policy": "lerobot/smolvla_libero",
    "libero_suite": "libero_goal",
    "libero_task_id": 0,
    # Revision and camera rename map from LeRobot docs (docs/source/simulation.mdx at the pinned commit).
    "libero_policy_revision": "a24f1f380461b665158d52acd140152c79e8b36d",
    "libero_rename_map": {"observation.images.image": "observation.images.camera1",
                          "observation.images.image2": "observation.images.camera2"},
    "metaworld_task": "reach-v3",  # one of the MT10 tasks
    "vlm_name": "HuggingFaceTB/SmolVLM2-500M-Video-Instruct",  # backbone named in lerobot/smolvla_base config
    "mem_batch_size": 16,
    "mem_steps": 20,
    "mem_warmup_steps": 5,  # excluded from the sec/step average
    "dino_wm_osf_node": "bmw48",  # OSF project linked from the DINO-WM README
    "dino_wm_view_only": "a56a296ce3b24cceaf408383a175ce28",
    "dino_wm_file": "wall_single.zip",
}

WORK = Path.cwd()
RESULTS = WORK / "results"
RESULTS.mkdir(exist_ok=True)
(RESULTS / "config.json").write_text(json.dumps(CONFIG, indent=2))
os.environ["MUJOCO_GL"] = "egl"  # inherited by every step process
T_START = time.time()
DEVIATIONS = []

In [ ]:
def sh(cmd):
    """Run a shell command; stop the notebook if it fails."""
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

def selected(name):
    return CONFIG["steps_to_run"] == "all" or name in CONFIG["steps_to_run"]

def run_step(name, script, *args):
    """Run a step script with the chosen Python. It must write results/<name>.json; if not, record FAIL."""
    if not selected(name):
        print(name, "-> skipped (not in CONFIG['steps_to_run'])")
        return None
    out = RESULTS / f"{name}.json"
    out.unlink(missing_ok=True)
    # input="N\n" answers LIBERO's first-import question about a custom dataset path (keep defaults).
    p = subprocess.run([PY, script, *args], input="N\n", capture_output=True, text=True)
    print(p.stdout[-4000:])
    print(p.stderr[-4000:])
    if p.returncode != 0 or not out.exists():
        out.write_text(json.dumps({"status": "FAIL", "error": (p.stderr or p.stdout)[-1500:]}))
    result = json.loads(out.read_text())
    print(name, "->", result)
    return result

## 1. Install LeRobot (pinned commit) with libero + metaworld extras

In [ ]:
print(sys.version)
sh("nvidia-smi")

if sys.version_info >= (3, 12):
    PY = sys.executable
    PIP = f"{PY} -m pip install"
    UNINSTALL = f"{PY} -m pip uninstall -y"
else:
    # LeRobot needs Python >= 3.12. Approved fallback: a Python 3.12 venv made by uv.
    DEVIATIONS.append(f"Kaggle Python is {sys.version.split()[0]} (< 3.12); all steps ran in a uv-created Python 3.12 venv.")
    sh(f"{sys.executable} -m pip install uv=={CONFIG['uv_version']}")
    sh(f"{sys.executable} -m uv venv --python 3.12 /tmp/py312")  # in /tmp so it is not saved as notebook output
    PY = "/tmp/py312/bin/python"
    PIP = f"{sys.executable} -m uv pip install --python {PY}"
    UNINSTALL = f"{sys.executable} -m uv pip uninstall --python {PY}"

# Default PyPI torch wheels are CUDA 13 and need driver >= 580.65 (LeRobot install docs).
# For older drivers the docs say: torch from the cu126 index, torchcodec from the cpu index, then LeRobot.
driver = subprocess.run(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
                        capture_output=True, text=True, check=True).stdout.strip()
if tuple(int(x) for x in driver.split(".")[:2]) < (580, 65):
    sh(f"{PIP} --index-url https://download.pytorch.org/whl/cu126 torch torchvision")
    sh(f"{PIP} --index-url https://download.pytorch.org/whl/cpu torchcodec")

lerobot_src = f"git+https://github.com/huggingface/lerobot.git@{CONFIG['lerobot_commit']}"
sh(f'{PIP} "lerobot[libero,metaworld,smolvla] @ {lerobot_src}" "peft=={CONFIG["peft_version"]}"')

# Kaggle preinstalls torchao 0.10.0; peft rejects torchao < 0.16 when creating LoRA layers. We don't use it.
sh(f"{UNINSTALL} torchao")

(RESULTS / "step1_install.json").write_text(json.dumps({
    "status": "PASS", "python_used": PY, "driver": driver, "torchao_uninstalled": True,
    "install_minutes": round((time.time() - T_START) / 60, 1)}))

## 2. Versions and imports

In [ ]:
%%writefile step2_versions.py
import json, sys
import importlib.metadata as md
from pathlib import Path
import torch

r = {"python": sys.version.split()[0], "torch": torch.__version__, "torch_cuda": torch.version.cuda,
     "cuda_available": torch.cuda.is_available()}
r["gpu"] = torch.cuda.get_device_name(0) if r["cuda_available"] else None
for pkg in ["lerobot", "mujoco", "gymnasium", "metaworld", "hf-libero", "transformers", "peft"]:
    try:
        r[pkg] = md.version(pkg)
    except md.PackageNotFoundError:
        r[pkg] = "not installed"

import lerobot, libero, metaworld  # the step crashes (FAIL) if any import fails
r["imports_ok"] = True
r["status"] = "PASS" if r["cuda_available"] else "FAIL"
Path("results/step2_versions.json").write_text(json.dumps(r))

In [ ]:
run_step("step2_versions", "step2_versions.py")

## 3. LIBERO: `lerobot/smolvla_libero` on `libero_goal`, 1 episode

In [ ]:
%%writefile step3_libero.py
import json, os, shutil, subprocess, sys, time
from pathlib import Path

C = json.loads(Path("results/config.json").read_text())
out_dir = Path("outputs/feasibility/libero_eval")
shutil.rmtree(out_dir, ignore_errors=True)  # only our own previous output
# lerobot-eval lives next to the Python we run with
exe = shutil.which("lerobot-eval", path=f"{Path(sys.executable).parent}:{os.environ['PATH']}")

cmd = [exe,
       f"--policy.path={C['libero_policy']}",
       f"--policy.pretrained_revision={C['libero_policy_revision']}",
       "--policy.device=cuda",
       f"--rename_map={json.dumps(C['libero_rename_map'])}",
       "--env.type=libero",
       f"--env.task={C['libero_suite']}",
       f"--env.task_ids=[{C['libero_task_id']}]",
       "--eval.batch_size=1",
       "--eval.n_episodes=1",
       f"--seed={C['seed']}",
       f"--output_dir={out_dir}"]
print(" ".join(cmd), flush=True)
t0 = time.time()
subprocess.run(cmd, input="N\n", text=True, check=True)
wall_s = time.time() - t0  # includes policy download, env build and the episode

info = json.loads((out_dir / "eval_info.json").read_text())
print("eval_info keys:", list(info))
overall = info["overall"]
r = {"status": "PASS",  # PASS = eval ran end to end; success is reported separately
     "pc_success": overall["pc_success"],
     "n_episodes": overall["n_episodes"],
     "eval_s": overall.get("eval_s"),
     "wall_time_s": round(wall_s, 1)}
Path("results/step3_libero.json").write_text(json.dumps(r))

In [ ]:
run_step("step3_libero", "step3_libero.py")

## 4. Meta-World: reset one MT10 task, 10 random actions, save one frame

In [ ]:
%%writefile step4_metaworld.py
import json, random
from pathlib import Path
import numpy as np
import metaworld
from PIL import Image

C = json.loads(Path("results/config.json").read_text())
seed, name = C["seed"], C["metaworld_task"]
random.seed(seed)
np.random.seed(seed)

mt10 = metaworld.MT10(seed=seed)
env = mt10.train_classes[name](render_mode="rgb_array")
env.set_task(next(t for t in mt10.train_tasks if t.env_name == name))
obs, info = env.reset(seed=seed)
env.action_space.seed(seed)
for _ in range(10):
    obs, reward, terminated, truncated, info = env.step(env.action_space.sample())

frame = env.render()
Image.fromarray(frame).save("results/metaworld_frame.png")
r = {"status": "PASS", "task": name, "frame_shape": list(frame.shape), "obs_dim": int(obs.shape[0])}
Path("results/step4_metaworld.json").write_text(json.dumps(r))

In [ ]:
run_step("step4_metaworld", "step4_metaworld.py")

## 5. Memory test: SmolVLM2-500M backbone + small action expert, fp16 autocast + GradScaler

Two setups, each in its own process so peak numbers don't mix:
- **frozen**: the whole backbone is frozen and runs under `no_grad`. Only the action expert trains (main plan).
- **lora**: the vision encoder and connector are frozen. LoRA (r=16) goes on the language model's attention projections, plus the action expert (stretch plan).

The backbone weights are loaded in fp16 because frozen weights get no gradients. The trainable weights (expert, LoRA) stay fp32, as GradScaler needs. Inputs are random: one 512×512 image plus a short instruction per sample, and 50×7 action chunks.

In [ ]:
%%writefile step5_memory.py
import json, random, resource, sys, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor

SETUP = sys.argv[1]  # "frozen" or "lora"
C = json.loads(Path("results/config.json").read_text())
seed, B = C["seed"], C["mem_batch_size"]
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
CHUNK, ACT_DIM, DEV = 50, 7, "cuda"


class ActionExpert(nn.Module):
    """Small transformer: action-chunk queries cross-attend to backbone features, predict the chunk."""
    def __init__(self, vlm_dim, d=512, layers=6, heads=8):
        super().__init__()
        self.feat_in = nn.Linear(vlm_dim, d)
        self.act_in = nn.Linear(ACT_DIM, d)
        self.pos = nn.Parameter(torch.zeros(1, CHUNK, d))
        layer = nn.TransformerDecoderLayer(d, heads, 4 * d, batch_first=True)
        self.decoder = nn.TransformerDecoder(layer, layers)
        self.act_out = nn.Linear(d, ACT_DIM)

    def forward(self, feats, noisy_actions):
        q = self.act_in(noisy_actions) + self.pos
        return self.act_out(self.decoder(q, self.feat_in(feats)))


vlm = AutoModelForImageTextToText.from_pretrained(C["vlm_name"], torch_dtype=torch.float16).to(DEV)
vlm.requires_grad_(False)
if SETUP == "lora":
    from peft import LoraConfig, inject_adapter_in_model
    # Regex keeps LoRA on the language model only (vision encoder stays untouched).
    cfg = LoraConfig(r=16, lora_alpha=32, target_modules=r".*text_model.*\.(q_proj|k_proj|v_proj|o_proj)")
    vlm = inject_adapter_in_model(cfg, vlm)
    for n, p in vlm.named_parameters():
        p.requires_grad = "lora_" in n
        if p.requires_grad:
            p.data = p.data.float()  # trainable weights in fp32 for GradScaler

expert = ActionExpert(vlm.config.text_config.hidden_size).to(DEV)
params = [p for p in list(vlm.parameters()) + list(expert.parameters()) if p.requires_grad]
opt = torch.optim.AdamW(params, lr=1e-4)
scaler = torch.amp.GradScaler("cuda")

# One fixed random batch; only noise/targets change per step.
proc = AutoProcessor.from_pretrained(C["vlm_name"])
proc.image_processor.do_image_splitting = False  # one 512x512 image -> one tile, as in SmolVLA
rng = np.random.default_rng(seed)
images = [[Image.fromarray(rng.integers(0, 256, (512, 512, 3), dtype=np.uint8))] for _ in range(B)]
texts = ["<image>put the bowl on the plate"] * B
batch = proc(text=texts, images=images, return_tensors="pt", padding=True).to(DEV)
batch["pixel_values"] = batch["pixel_values"].half()

times = []
for step in range(C["mem_steps"]):
    torch.cuda.synchronize()
    t0 = time.time()
    noisy = torch.randn(B, CHUNK, ACT_DIM, device=DEV)
    target = torch.randn_like(noisy)
    with torch.autocast("cuda", dtype=torch.float16):
        with torch.set_grad_enabled(SETUP == "lora"):
            feats = vlm.model(**batch).last_hidden_state
        loss = nn.functional.mse_loss(expert(feats, noisy).float(), target)
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.step(opt)
    scaler.update()
    torch.cuda.synchronize()
    times.append(time.time() - t0)

r = {"status": "PASS",
     "setup": SETUP,
     "trainable_params_M": round(sum(p.numel() for p in params) / 1e6, 2),
     "backbone_tokens_per_sample": int(batch["input_ids"].shape[1]),
     "peak_gpu_allocated_GB": round(torch.cuda.max_memory_allocated() / 1024**3, 2),
     "peak_gpu_reserved_GB": round(torch.cuda.max_memory_reserved() / 1024**3, 2),
     "peak_ram_GB": round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024**2, 2),  # ru_maxrss is KB on Linux
     "sec_per_step": round(float(np.mean(times[C["mem_warmup_steps"]:])), 3),
     "final_loss": round(loss.item(), 4)}
Path(f"results/step5_{SETUP}.json").write_text(json.dumps(r))

In [ ]:
run_step("step5_frozen", "step5_memory.py", "frozen")
run_step("step5_lora", "step5_memory.py", "lora")

## 6. DINO-WM Wall dataset: link and size only (no download)

In [ ]:
%%writefile step6_dinowm.py
import json, urllib.request
from pathlib import Path

C = json.loads(Path("results/config.json").read_text())
vo = C["dino_wm_view_only"]

def walk(url):
    """Yield every file entry under an OSF folder listing, following pages and subfolders."""
    while url:
        page = json.load(urllib.request.urlopen(url, timeout=30))
        for f in page["data"]:
            if f["attributes"]["kind"] == "folder":
                yield from walk(f["relationships"]["files"]["links"]["related"]["href"])
            else:
                yield f
        url = page["links"].get("next")

root = f"https://api.osf.io/v2/nodes/{C['dino_wm_osf_node']}/files/osfstorage/?view_only={vo}"
match = [f for f in walk(root) if f["attributes"]["name"] == C["dino_wm_file"]]
f = match[0]  # IndexError -> FAIL if the file is not listed
size = f["attributes"]["size"]
r = {"status": "PASS", "file": f["attributes"]["materialized_path"], "download_link": f["links"]["download"],
     "size_bytes": size, "size_GB": round(size / 1e9, 2) if size is not None else "unknown"}
Path("results/step6_dinowm.json").write_text(json.dumps(r))

In [ ]:
run_step("step6_dinowm", "step6_dinowm.py")

## 7. Write results/feasibility.md (numbers come only from the step JSON files)

In [ ]:
STEPS = [("1. Install", "step1_install"), ("2. Versions/imports", "step2_versions"),
         ("3. LIBERO eval", "step3_libero"), ("4. Meta-World", "step4_metaworld"),
         ("5a. Memory: frozen backbone", "step5_frozen"), ("5b. Memory: LoRA", "step5_lora"),
         ("6. DINO-WM Wall", "step6_dinowm")]
total_min = round((time.time() - T_START) / 60, 1)

lines = ["# Feasibility check (Kaggle)", "",
         f"- Date (UTC): {time.strftime('%Y-%m-%d %H:%M', time.gmtime())}",
         f"- Seed: {CONFIG['seed']}, LeRobot commit: `{CONFIG['lerobot_commit']}`",
         f"- Steps run: {CONFIG['steps_to_run']}",
         f"- Total notebook time: {total_min} min (budget 30 min: {'OK' if total_min <= 30 else 'OVER'})",
         "", "| Step | Status | Numbers |", "|---|---|---|"]
errors = []
for label, name in STEPS:
    path = RESULTS / f"{name}.json"
    if name != "step1_install" and not selected(name):
        r = {"status": "SKIPPED"}  # never reuse an older result file
    else:
        r = json.loads(path.read_text()) if path.exists() else {"status": "NOT RUN"}
    nums = ", ".join(f"{k}={v}" for k, v in r.items() if k not in ("status", "error"))
    lines.append(f"| {label} | {r['status']} | {nums} |")
    if "error" in r:
        errors.append(f"### {label}\n```\n{r['error']}\n```")

lines += ["", "## Deviations", *(f"- {d}" for d in DEVIATIONS or ["None."])]
lines += ["", "## Notes",
          "- Step 3 PASS means the eval ran end to end; with 1 episode, pc_success is 0 or 100.",
          "- Step 5 sec/step averages steps after the warm-up steps in config.",
          "- Step 5 peak RAM is per process (ru_maxrss)."]
if errors:
    lines += ["", "## Errors", *errors]

(RESULTS / "feasibility.md").write_text("\n".join(lines) + "\n")
print("\n".join(lines))